# RAG Walkthrough

Discover restaurants in Bengaluru and get answers about cuisines, menus, pricing, and more—using a RAG-powered assistant.


## 1. Project setup

Find the project folder and import the functions we’ll use to prepare chunks, create LangChain documents, generate embeddings, and connect to Pinecone.

This cell only loads the functions; it does not call any APIs or change the data.

Select the `.venv` Python kernel, then run the cell with **Shift+Enter**.


In [ ]:
import json
import sys
from pathlib import Path

# Find the project whether the notebook starts here or in a subfolder.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/rag_store.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the GenAcademy project folder.")
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from prepare_chunks import prepare_chunks
from rag_store import as_document, load_chunks, get_embeddings, existing_store, get_pinecone

print("Ready. Python:", sys.executable)


## 2. Load data and create chunks

Load `restaurants.json` and `faqs.json`, split the data into chunks, and save them to `data/chunks.json`. Running this cell replaces the file if it already exists.


In [ ]:
restaurants = json.loads((ROOT / "data/restaurants.json").read_text())
faqs = json.loads((ROOT / "data/faqs.json").read_text())

chunks = prepare_chunks(restaurants, faqs, max_chars=3000)

output_path = ROOT / "data/chunks.json"
output_path.write_text(json.dumps(chunks, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print(f"{len(restaurants)} restaurants + {len(faqs)} FAQs → {len(chunks)} chunks")
print(f"Saved to: {output_path}")


## 3. Embed the data

Convert each chunk into a LangChain Document, then use Nebius to generate one embedding vector per chunk. Each vector contains 4,096 numbers.

This cell calls the API and uses credits. It keeps the vectors in notebook memory; it does not upload them to Pinecone. Running it again generates the embeddings again.


In [ ]:
documents = [as_document(chunk) for chunk in chunks]
embeddings = get_embeddings()
vectors = []
batch_size = 10

for start in range(0, len(documents), batch_size):
    batch = documents[start:start + batch_size]
    batch_vectors = embeddings.embed_documents([doc.page_content for doc in batch])
    if len(batch_vectors) != len(batch) or any(len(v) != 4096 for v in batch_vectors):
        raise ValueError("Unexpected number or size of embeddings returned by Nebius.")
    vectors.extend(batch_vectors)
    print(f"Embedded {len(vectors)}/{len(documents)} chunks")

print(f"Created {len(vectors)} vectors, each containing 4,096 numbers.")
if vectors:
    print("First vector (first 5 numbers):", vectors[0][:5])


## 4. Store embeddings in Pinecone

Store each embedding alongside its original chunk and metadata so we can search it later. The index is created if missing, or reused if it already exists. This step uses the embeddings from section 3 without embedding again.


In [ ]:
import importlib
import push_vectors as pinecone_upload

# Reload the helper so fixes apply without losing the embeddings in memory.
importlib.reload(pinecone_upload)

manifest = pinecone_upload.push_vectors(chunks, documents, vectors, embedding_model=embeddings.model)
store = existing_store()

print("Index:", manifest["index"])
print("Namespace:", manifest["namespace"])
print("Stored vectors:", manifest["chunk_count"])


## 5. Interpret the question and retrieve context

The LLM interprets your question and extracts a validated query plan. Python computes counts and complete lists from the full dataset. Other questions use hybrid retrieval. This step uses API credits for planning and, for RAG questions, search.


In [ ]:
from hybrid_retriever import HybridRetriever
import importlib
import structured_search
importlib.reload(structured_search)
import rag_chat
importlib.reload(rag_chat)
from rag_chat import build_context, get_chat_model
from structured_search import structured_context

question = input("Ask a question: ").strip()

hybrid = HybridRetriever(store=store, documents=documents)
planner_llm = get_chat_model()
structured = structured_context(question, llm=planner_llm)
if structured is not None:
    sources, context = structured
    print("Using the full restaurant dataset")
else:
    results = hybrid.search(question, k=5)
    sources, context = build_context(results)

print(f"Retrieved {len(sources)} chunks for: {question}")
for source in sources:
    print(f"[{source['label']}] {source['title']} — {source['chunk_id']}")

print("\nContext for the chat model:")
print(json.dumps(sources, ensure_ascii=False, indent=2))


## 6. Build the prompt

Combine the system instructions, your question, and the context retrieved in section 5. Display the exact messages that will be sent to the chat model.

The system instructions are in `prompts/restaurant_system.txt`. This step runs locally; it does not call the chat model.


In [ ]:
from rag_chat import build_answer_prompt

prompt, parser = build_answer_prompt()
prompt_inputs = {
    "question": question,
    "context": context,
    "history": [],
    "filters": "{}",
}
prompt_messages = prompt.invoke(prompt_inputs).to_messages()
prompt_sources = json.loads(context)

for message in prompt_messages:
    print(f"\n--- {message.type.upper()} ---\n{message.content}")


## 7. Generate an answer

Send the prompt from section 6 to the Nebius chat model through LangChain. Check that the answer’s citations refer to the retrieved sources.

This step uses API credits. It reuses the context already retrieved; it does not run search again.

Structured counts and lists are rendered directly from the dataset without a chat-model call. Other questions use the Nebius chat model.


In [ ]:
from rag_chat import get_chat_model, generate_answer

llm = get_chat_model()
answer = generate_answer(prompt_messages, prompt_sources, llm, parser)

print(answer.answer)
print("Status:", answer.status)


## 8. Review the sources

Read the original chunks cited in the answer. Compare the answer with these records to check whether the information is supported.

Citation validation checks source labels; it does not guarantee that every claim is correct. No API calls are made here.


In [ ]:
cited_sources = [source for source in prompt_sources if source["label"] in answer.citations]

if not cited_sources:
    print("No sources were cited for this response.")

for source in cited_sources:
    print(f"\n[{source['label']}] {source['title']} — {source['chunk_id']}")
    print(json.dumps(source["content"], ensure_ascii=False, indent=2))


To try another question, edit section 5 and rerun **5 → 6 → 7 → 8**. You do not need to repeat data preparation or embedding.

This walkthrough tests one question at a time. For a conversation with follow-ups, run `.venv/bin/python -m streamlit run app.py` in the terminal.


### Run the complete LangGraph workflow

Sections 5–8 show the individual components. This cell runs the complete app workflow again: resolve follow-up → plan → structured lookup or retrieval → answer → evidence check → optional handoff. It uses fresh API calls. Reuse `graph_bot` for follow-ups. History is in memory; no persistent checkpointer is configured. Demo handoffs are saved locally.


In [ ]:
import importlib
import rag_chat
importlib.reload(rag_chat)
graph_bot = rag_chat.RestaurantChat()
graph_reply = graph_bot.ask(question)
print(graph_reply["answer"])
print("Steps:", " → ".join(graph_reply["graph_steps"]))
print("Evidence check:", graph_reply["support_assessment"])


## 9. Evaluate the assistant

First review the 22 questions and expected answers. This preview runs locally.

The evals cover facts, menus, filters, missing information, demo values, unsupported requests, and follow-ups.


In [ ]:
import pandas as pd
import importlib
import structured_search
import rag_chat
import run_evals as eval_module

# Refresh updated helpers without restarting the notebook kernel.
importlib.reload(structured_search)
importlib.reload(rag_chat)
importlib.reload(eval_module)
from run_evals import load_cases, run_evals, review_rows, record_review

eval_cases = load_cases()
display(pd.DataFrame([
    {"id": case["id"], "category": case["category"],
     "question": case["question"], "expected": case["expected_answer"]}
    for case in eval_cases
]))


Run the cell below to evaluate the real assistant. Complete sections 1–4 first. This uses Nebius/Pinecone API credits; the follow-up case makes extra calls.

Results are saved to `evals/runs/`. Automatic checks are screening signals, not proof that an answer is correct. Review each answer and its sources yourself.


In [ ]:
# Use run_evals(limit=3) if you want to try a small run first.
eval_report = run_evals()
print("Results saved to:", eval_report["report_path"])
print(json.dumps(eval_report["summary"], indent=2))
display(pd.DataFrame(review_rows(eval_report)))


### Load saved eval results

If the eval run was interrupted or the kernel restarted, run this cell to load the most recently saved report. This does not call any APIs or rerun evals.


In [ ]:
import json
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "evals").is_dir():
    project_root = project_root.parent
reports = list((project_root / "evals" / "runs").glob("*.json"))
if not reports:
    raise FileNotFoundError("No saved eval reports found. Run the evaluation cell above first.")

latest_report = max(reports, key=lambda path: path.stat().st_mtime)
eval_report = json.loads(latest_report.read_text())
print("Loaded:", latest_report.name)
print("Saved results:", len(eval_report["results"]))

result_index = 6  # Seventh result; change to 0 for the first result.
if len(eval_report["results"]) > result_index:
    print(json.dumps(eval_report["results"][result_index], indent=2, ensure_ascii=False))
else:
    print("The seventh result is not saved yet. Choose a smaller result_index if results are available.")


To inspect a result, run `eval_report["results"][0]`. It contains the expected answer, actual conversation, retrieved sources, and individual checks.

After reviewing a case, use `record_review(report_path, case_id, correctness=..., grounding=..., appropriate_behavior=..., resolved_without_handoff=..., notes=...)` with your own True/False ratings. Human-review scores stay empty until you provide ratings. See `evals/README.md` for the rubric.


### Review and export

Open **Evaluation results** in Streamlit to compare expected and actual answers. Use `record_review()` to save human judgments if needed. Run the cell below to export a report. Unreviewed metrics remain unmeasured.


In [ ]:
from eval_report import export_report
report_document = export_report(eval_report["report_path"])
print("Evaluation report:", report_document)
